Explore the escape effect

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

In [2]:
%load_ext autoreload
from behave_analysis.process.process import Process
from behave_analysis.utils.creating_directories import make_directory

import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from rastermap import Rastermap, utils
from scipy.stats import zscore
%matplotlib inline

In [11]:
session.barrier_location[0]

[224, 512]

In [9]:
"""Make rastermaps with variable time of escape to see arrival at shelter"""
for exp in experiments_objects[1:]:
    nickname = exp.nick_name + '_' + exp.experiment_date + '_fullescaperoute'
    print(nickname)
    session, frame_by_cluster_matrix, behave, y_pos, x_pos = load(exp)
    esc_speed, esc_y, escape_matrix, start = extract_escape_time(session, frame_by_cluster_matrix, behave, y_pos, x_pos)
    isort, fit_spks = run_rastermap(escape_matrix)
    plot_rastermap(fit_spks, esc_speed, esc_y, session, isort, nickname, start)

JAL004_2023_09_19_fullescaperoute
2024-12-12 14:44:40,984 [INFO] normalizing data across axis=1
2024-12-12 14:44:40,996 [INFO] projecting out mean along axis=0
2024-12-12 14:44:41,008 [INFO] data normalized, 0.02sec
2024-12-12 14:44:41,009 [INFO] sorting activity: 144 valid samples by 3702 timepoints
2024-12-12 14:44:42,027 [INFO] n_PCs = 64 computed, 1.04sec
2024-12-12 14:44:42,032 [INFO] skipping clustering, n_clusters is None
2024-12-12 14:45:50,155 [INFO] clusters sorted, time 69.17sec
2024-12-12 14:45:50,180 [INFO] rastermap complete, time 69.20sec
JAL004_2023_08_28_fullescaperoute
2024-12-12 14:46:23,137 [INFO] normalizing data across axis=1
2024-12-12 14:46:23,141 [INFO] projecting out mean along axis=0
2024-12-12 14:46:23,147 [INFO] data normalized, 0.01sec
2024-12-12 14:46:23,148 [INFO] sorting activity: 170 valid samples by 720 timepoints
2024-12-12 14:46:23,496 [INFO] n_PCs = 64 computed, 0.36sec
2024-12-12 14:46:23,498 [INFO] skipping clustering, n_clusters is None
2024-12-

In [ ]:
"""Make rastermaps -1 to +5 sec from puff"""
for exp in experiments_objects:
    nickname = exp.nick_name + '_' + exp.experiment_date
    print(nickname)
    session, frame_by_cluster_matrix, behave, y_pos, _ = load(exp)
    esc_speed, esc_y, escape_matrix = extract_time(session, frame_by_cluster_matrix, behave, y_pos)
    isort, fit_spks = run_rastermap(escape_matrix)
    plot_rastermap(fit_spks, esc_speed, esc_y, session, isort, nickname)

In [3]:
def load(exp):
    # load session
    session = Process(exp).load_session()
    base_path = os.path.join(session.base_path, session.processed_path)

    # spikeys
    # spike_data = pl.read_csv(os.path.join(base_path, "good_spike_data.csv"))

    # matrix
    frame_by_cluster_matrix = np.load(
                os.path.join(session.base_path, session.processed_path) + "\\" + "frame_by_good_cluster_matrix.npy")

    # behavior
    video_df = pl.read_csv(os.path.join(base_path, "full_video_dataframe.csv"))
    behave = video_df['speed'].to_numpy()
    y_pos = video_df['mouse_y_position'].to_numpy()
    x_pos = video_df['mouse_x_position'].to_numpy()
    return session, frame_by_cluster_matrix, behave, y_pos, x_pos

In [4]:
def extract_time(session, frame_by_cluster_matrix, behave, y_pos):
    # extract the time around escapes
    for tr, of in enumerate(session.audio.onset_frames):
        if isinstance(of, list): of = of[0]
        if isinstance(of, np.ndarray): of = of[0]
        this_esc = frame_by_cluster_matrix[of - 40:of + (5*40),:]
        if tr == 0:
            escape_matrix = this_esc
            esc_speed = behave[of - 40:of + (5*40)]
            esc_y = y_pos[of - 40:of + (5*40)]
        else:
            escape_matrix = np.vstack((escape_matrix,this_esc))
            esc_speed = np.append(esc_speed, behave[of - 40:of + (5*40)])
            esc_y = np.append(esc_y, y_pos[of - 40:of + (5*40)])
    return esc_speed, esc_y, escape_matrix

In [5]:
def extract_escape_time(session, frame_by_cluster_matrix, behave, y_pos, x_pos):
    # extract the time around escapes
    start = [40]
    for tr, of in enumerate(session.audio.onset_frames):
        if isinstance(of, list): of = of[0]
        if isinstance(of, np.ndarray): of = of[0]
        y_loc = y_pos[of:of + (20*40)]
        x_loc = x_pos[of:of + (20*40)]
        in_shelt_y = y_loc > session.shelter_location[0][1]
        in_shelt_x = np.logical_and(x_loc > session.shelter_location[0][0],x_loc < session.shelter_location[1][0])
        in_shelt = np.where(np.logical_and(in_shelt_x, in_shelt_y))[0]
        if len(in_shelt) == 0: in_shelt = 5*40 # cases when mouse never reaches shelter
        else: in_shelt = in_shelt[0]+40
        if tr == 0:
            escape_matrix = frame_by_cluster_matrix[of - 40:of + in_shelt,:]
            esc_speed = behave[of - 40:of + in_shelt]
            esc_y = y_pos[of - 40:of + in_shelt]
        else:
            escape_matrix = np.vstack((escape_matrix,frame_by_cluster_matrix[of - 40:of + in_shelt,:]))
            esc_speed = np.append(esc_speed, behave[of - 40:of + in_shelt])
            esc_y = np.append(esc_y, y_pos[of - 40:of + in_shelt])
        start.append(int(in_shelt + start[-1] + 40))
    return esc_speed, esc_y, escape_matrix, start[:-1]

In [6]:
def run_rastermap(escape_matrix):
  # rastermap with running speed and y position

  spks = zscore(escape_matrix.T, axis=1) # need neurons x time
  intereting_neurons = np.mean(spks[:,:40], axis = 1) < np.mean(spks[:,40:6*40], axis = 1)
  fit_spks = spks[intereting_neurons,:]
  if np.shape(fit_spks)[0] > 200:
    sorter = np.argsort(np.mean(fit_spks[:,40:4*40], axis = 1) - np.mean(fit_spks[:,:40], axis = 1))
    fit_spks = fit_spks[sorter,:]
    fit_spks = fit_spks[-200:,:]
  # fit_spks = fit_spks[:,:6*40]

  model = Rastermap(n_clusters=None, # None turns off clustering and sorts single neurons 
                    n_PCs=64, # use fewer PCs than neurons
                    locality=0.15, # some locality in sorting (this is a value from 0-1)
                    time_lag_window=40, # use future timepoints to compute correlation
                    grid_upsample=0, # 0 turns off upsampling since we're using single neurons
                  ).fit(fit_spks)
  y = model.embedding # neurons x 1
  isort = model.isort
  return isort, fit_spks

In [7]:
def plot_rastermap(fit_spks, esc_speed, esc_y, session, isort, nickname, start = []):
    # make figure with grid for easy plotting
    fig = plt.figure(figsize=(12,6), dpi=200)
    grid = plt.GridSpec(9, 20, figure=fig, wspace = 0.05, hspace = 0.3)

    # plot running speed
    ax = plt.subplot(grid[0, :-1])
    ax.plot(esc_speed, color='r')
    ax.axis("off")

    # plot y position
    ax2 = ax.twinx()
    ax2.plot(esc_y, color = 'b')
    ax2.set_xlim((0,len(esc_speed)))
    ax2.axis("off")

    # plot superneuron activity
    ax = plt.subplot(grid[1:, :-1])
    ax.imshow(fit_spks[isort,:], cmap="gray_r", vmin=0, vmax=1.2, aspect="auto")
    if len(start) == 0:
        start = 40
        for i in np.arange(len(session.audio.onset_frames)):
            ax.plot([start,start],[0,len(isort)],'--b')
            start += 6*40
    else:
        for i in start:
            ax.plot([i,i],[0,len(isort)],'--b')
    ax.set_xlabel("time")
    ax.set_ylabel("neurons")

    stim_resp_path = make_directory(os.path.join(session.base_path, session.processed_path, "stim_resp", "rastermap"))
    save_path=str(stim_resp_path) + "/" + "good_cluster_rastermap_trial.png"
    fig.savefig(save_path)

    dump_path = "Z:\Jasmine_Laurence\escape_rastermap"
    fig.savefig(dump_path + "/" + nickname + ".png")
    plt.close()